# Tutorial: Mesh generation with Gmsh and solving the Poisson equation in Gridap

## Creating a triangulation of the unit square using Gmsh

First, we construct a simple triangular mesh on the unit square $\Omega = (0,1)^2$ using Gmsh. Start by loading and initializing the `Gmsh` package in Julia.

In [1]:
# import Pkg; Pkg.add("Gmsh")
import Gmsh: gmsh;
gmsh.initialize();

Next, we add the four vertices of the square. They are specified by $(x,y,z)$ coordinates.

In [2]:
gmsh.model.add("unit_square");
h = 0.05; # target mesh size
gmsh.model.geo.addPoint(0.0, 0.0, 0.0, h, 1); # last argument is optional identifier, unique per dimension
gmsh.model.geo.addPoint(1.0, 0.0, 0.0, h, 2);
gmsh.model.geo.addPoint(1.0, 1.0, 0.0, h, 3);
gmsh.model.geo.addPoint(0.0, 1.0, 0.0, h, 4);

We define the edges of the squre:

In [3]:
gmsh.model.geo.addLine(1, 2, 1); 
gmsh.model.geo.addLine(2, 3, 2); # line 2 goes from point 2 to point 3
gmsh.model.geo.addLine(3, 4, 3);
gmsh.model.geo.addLine(4, 1, 4);

The next step is to add a so called curve loop. From the Gmsh tutorial:
> In order to define a simple rectangular surface from the four curves defined above, a curve loop has first to be defined. A curve loop is defined by an ordered list of connected curves, a sign being associated with each curve (depending on the orientation of the curve to form a loop). The API function to create curve loops takes a list of integers as first argument, and the curve loop tag (which must be unique amongst curve loops) as the second (optional) argument.

In [4]:
gmsh.model.geo.addCurveLoop([1, 2, 3, 4], 1);

The surface (the unit square) is then defined as a list of curve loops. Here, there is only one since we have a simple surface without any holes in it.

In [5]:
gmsh.model.geo.addPlaneSurface([1], 1);

What we have created here are CAD entities defining the *domain/surface*. We must now synchronize them with the Gmsh model to allow for *meshing*.

In [6]:
gmsh.model.geo.synchronize();

In order to use this in a PDE solver, we must specify the boundary $\Gamma$ of $\Omega$. We can divide it into several parts if we want to use so called *mixed* boundary conditions. Here, we choose to have a simple boundary encircling the square. We put all edges into a `PhysicalGroup` and call it *boundary*. We use the identifier *5* since we have used the identifiers *1,2,3* and *4* for the edges. To make sure the corners are included, we need to add them separately. We then name the surface we created the *domain*.

In [7]:
# Define physical groups without the string argument
edges_tag = gmsh.model.addPhysicalGroup(1, [1, 2, 3, 4])   # edges
corners_tag = gmsh.model.addPhysicalGroup(0, [1, 2, 3, 4]) # corners
domain_tag = gmsh.model.addPhysicalGroup(2, [1])           # surface

# Set names for the physical groups
gmsh.model.setPhysicalName(1, edges_tag, "boundary")
gmsh.model.setPhysicalName(0, corners_tag, "boundary")
gmsh.model.setPhysicalName(2, domain_tag, "domain")

Now we can generate the mesh with the specified target mesh size.

In [8]:
gmsh.model.mesh.generate(2); # dimension is 2

Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 30%] Meshing curve 2 (Line)
Info    : [ 60%] Meshing curve 3 (Line)
Info    : [ 80%] Meshing curve 4 (Line)
Info    : Done meshing 1D (Wall 0.000942653s, CPU 0.000277s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.0146936s, CPU 0.004305s)
Info    : 513 nodes 1028 elements


And save it to disk.

In [9]:
gmsh.write("unit_square.msh");
gmsh.finalize();

Info    : Writing 'unit_square.msh'...
Info    : Done writing 'unit_square.msh'


## Importing the mesh into Gridap

Next, we load the Gridap, Gridap.Io and GridapGmsh packages. Then, we create a Gridap compatible model and save it to disk. We also save the model as a vtk files so that we can visualize it in Paraview.

In [11]:
# import Pkg; Pkg.add("Gridap")
# import Pkg; Pkg.add("GridapGmsh")
using Gridap
using Gridap.Io
using GridapGmsh

model = GmshDiscreteModel("unit_square.msh");
writevtk(model,"model");

Info    : Reading 'unit_square.msh'...
Info    : 9 entities
Info    : 513 nodes
Info    : 1028 elements
Info    : Done reading 'unit_square.msh'


Here is the result. We have colored the mesh by the boundary. 

![mesh](unit_square.png)

## The Finite Element Space
With the mesh complete, we define a finite element (FE) space on $\Omega$ using first-order elements. This space will approximate the solution to our Poisson equation.

In [12]:
Ω = Triangulation(model)  # create triangulation
order = 1  # FE order
reffe = ReferenceFE(lagrangian, Float64, order)  # Reference FE
V = FESpace(model, reffe; conformity=:H1)  # Piecewise linear polynomials on model

UnconstrainedFESpace()

Next, we extract a nodal basis function. We export it to visualize it in VTK format.

In [13]:
ndofs = num_free_dofs(V)  # DOFs (nodes)
coeffs = zeros(ndofs)  
coeffs[rand(1:ndofs)] = 1.0  # Pick random node
phi = FEFunction(V, coeffs)  # Create FE function
writevtk(Ω, "basis_function", cellfields=["phi" => phi]) 

(["basis_function.vtu"],)

![basis-function](basis_function.png)

We define a function $v(x) = \sin(\pi x_1) \sin(\pi x_2)$ and interpolate it onto the FE space $V$.

In [15]:
v(x) = sin(pi * x[1]) * sin(pi * x[2]) 
I_h_v = interpolate(v, V)
writevtk(Ω, "I_h_v", cellfields=["I_h_v" => I_h_v])

(["I_h_v.vtu"],)

![interpolant](interpolant.png)

To compute the $L^2$ projection $P_h v$, we define the $L^2$ inner product as a bilinear form approximate the linear form by high order quadrature.

In [17]:
degree = 3 * order  # High-order quadrature
dΩ = Measure(Ω, degree)
a(u, w) = ∫(u * w) * dΩ  # Bilinear form
l(w) = ∫(v * w) * dΩ  # Linear form, v as before
op = AffineFEOperator(a, l, V, V)  
P_h_v = solve(op)
writevtk(Ω, "P_h_v", cellfields=["P_h_v" => P_h_v])  # Export to VTK

(["P_h_v.vtu"],)

![projection](projection.png)

It's hard to see a difference between this and $I_h v$.

## Solving the Poisson equation by finite elements

We now set up the Poisson problem with a manufactured solution $u(x) = x_1 + x_2$ which leads to $f(x) = 0$ and Dirichlet boundary conditions on $\partial \Omega$.

We define the weak formulation with bilinear form $a(u,v) = \int \nabla v \cdot \nabla u \, d\Omega$ and linear form $b(v) = \int f \, v \, d\Omega$.

In [18]:
u(x) = x[1] + x[2]  
f(x) = 0  # source term
V0 = TestFESpace(model, reffe, conformity=:H1, dirichlet_tags="boundary") # hom. BCs
U = TrialFESpace(V0, u)  # trial space with nonhom. BCs
degree = 2
dΩ = Measure(Ω, degree)
a(u, v) = ∫( ∇(v)⋅∇(u) ) * dΩ  # bilinear form
b(v) = ∫( v * f ) * dΩ  # linear form
# solve Poisson problem
op = AffineFEOperator(a, b, U, V0)
uh = solve(op)  

SingleFieldFEFunction():
 num_cells: 944
 DomainStyle: ReferenceDomain()
 Triangulation: BodyFittedTriangulation()
 Triangulation id: 9741289157359250162

In [19]:
e = u - uh
writevtk(Ω,"uh",cellfields=["uh" => uh])
el2 = sqrt(sum( ∫( e*e )*dΩ ))

8.198541736166448e-16

The $L^2$-norm $\|e\|_{L^2}$ is close to machine accuracy, as $u$ can be exactly represented in $U$. Now, we again construct a solution $u(x) = \sin(\pi x_1) \sin(\pi x_2)$, which leads to $f(x) = 2\pi^2 \sin(\pi x_1) \sin(\pi x_2)$, but now this can not be exactly represented in $U$.

In [20]:
u(x) = sin(pi * x[1]) * sin(pi * x[2])  # exact solution
f(x) = 2 * pi^2 * sin(pi * x[1]) * sin(pi * x[2])  # source term
order = 1  # experiment with changing this
reffe = ReferenceFE(lagrangian, Float64, order)
V0 = TestFESpace(model, reffe, conformity=:H1, dirichlet_tags="boundary")
U = TrialFESpace(V0, u) 
degree = 2*order
Ω = Triangulation(model)
dΩ = Measure(Ω, degree)
a(u, v) = ∫( ∇(v) ⋅ ∇(u) ) * dΩ  # Bilinear form
b(v) = ∫( v * f ) * dΩ  # linear form
op = AffineFEOperator(a, b, U, V0)  
uh = solve(op)  # solve variational problem
e = u - uh
el2 = sqrt(sum(∫( e * e ) * dΩ))  # L2 error norm
println("L2 error norm: $el2")
writevtk(Ω, "e", cellfields=["e" => e])  # Output results to VTK


L2 error norm: 0.0016450536388302067


(["e.vtu"],)

The error is larger than machine accuracy, as can be seen below:

![error](e.png)

## Vector valued finite elements

This is how to set up a vector-valued finite element space on $\Omega$ and applying Dirichlet boundary conditions. Note that the conditions are enforced on all components of the vector valued functions here.

In [21]:
reffe = ReferenceFE(lagrangian, VectorValue{2,Float64}, order)
V = TestFESpace(model,reffe;
  conformity=:H1,
  dirichlet_tags=["boundary"], 
  dirichlet_masks=[(true,true)])  # Apply Dirichlet condition to all components on this boundary

UnconstrainedFESpace()

That's it for now! Next, you can have a look at the [official Gridap tutorial notebooks](https://gridap.github.io/Tutorials/stable/#How-to-run-the-notebooks-locally-1)!